# Rebuild the Modelling Table

In [10]:

import numpy as np
import pandas as pd

RANDOM_STATE = 42
TARGET = "salary_in_usd"

df = pd.read_csv("../data/ds_salaries.csv")


In [11]:
text_cols = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size"
]

# Strip leading and trailing spaces
for col in text_cols:
    df[col] = df[col].str.strip()

# Normalize text labels
for col in text_cols:
    if col != "job_title":
        df[col] = df[col].str.upper()

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Separate features and target
X = df.drop(columns=[TARGET])  
y = df[TARGET]

print(X.shape, y.shape)


(2584, 8) (2584,)


# Split the Data

In [12]:

from sklearn.model_selection import train_test_split

# Split features and target into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=X["experience_level"]  
)

# Check split sizes
print(len(X_train), len(X_test))


2067 517


# Build the Preprocessing Pipeline

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

EXP_ORDER = ["EN", "MI", "SE", "EX"]  # junior -> executive

numeric_cols = ["work_year"]

onehot_low = [
    "employment_type",
    "company_size",
    "remote_ratio"
]

onehot_high = [
    "job_title",
    "employee_residence",
    "company_location"
]


In [14]:
def make_preprocessor(scale: bool) -> ColumnTransformer:
    exp_enc = OrdinalEncoder(categories=[EXP_ORDER])

    if scale:  # Linear models and KNN
        exp_enc = Pipeline([
            ("ord", exp_enc),
            ("sc", StandardScaler())
        ])

    return ColumnTransformer([
        ("exp", exp_enc, ["experience_level"]),

        (
            "num",
            StandardScaler() if scale else "passthrough",
            numeric_cols
        ),

        (
            "cat_low",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            onehot_low
        ),

        # Rare categories (< 1% of TRAINING rows) are pooled automatically
        (
            "cat_high",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=0.01,
                sparse_output=False
            ),
            onehot_high
        )
    ])

# Define the Candidate Models


In [15]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression,
    RidgeCV,
    LassoCV,
    ElasticNetCV,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [16]:
# Random State
RS = RANDOM_STATE


# Pipeline Factory
def make_pipe(model, scale=False):
    return Pipeline(
        [
            ("pre", make_preprocessor(scale)),
            ("model", model),
        ]
    )

In [17]:
# Candidate Models
candidates = {
    # Baseline
    "Dummy (median)": make_pipe(
        DummyRegressor(strategy="median")
    ),

    # Linear Models
    "Linear Regression": make_pipe(
        LinearRegression(),
        scale=True,
    ),

    "Ridge (CV)": make_pipe(
        RidgeCV(
            alphas=np.logspace(-2, 3, 30)
        ),
        scale=True,
    ),

    "Lasso (CV)": make_pipe(
        LassoCV(
            cv=5,
            max_iter=20000,
            random_state=RS,
        ),
        scale=True,
    ),

    "ElasticNet (CV)": make_pipe(
        ElasticNetCV(
            l1_ratio=[0.2, 0.5, 0.8],
            cv=5,
            max_iter=20000,
            random_state=RS,
        ),
        scale=True,
    ),

    # Distance-Based Model
    "KNN": make_pipe(
        KNeighborsRegressor(
            n_neighbors=15,
            weights="distance",
        ),
        scale=True,
    ),

    # Tree-Based Models
    "Decision Tree": make_pipe(
        DecisionTreeRegressor(
            max_depth=8,
            min_samples_leaf=10,
            random_state=RS,
        )
    ),

    "Random Forest": make_pipe(
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RS,
        )
    ),

    # Boosting Models
    "Gradient Boosting": make_pipe(
        GradientBoostingRegressor(
            random_state=RS,
        )
    ),

    "HistGradientBoosting": make_pipe(
        HistGradientBoostingRegressor(
            random_state=RS,
        )
    ),

    "XGBoost": make_pipe(
        XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
            random_state=RS,
        )
    ),

    "LightGBM": make_pipe(
        LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=15,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            verbose=-1,
            n_jobs=-1,
            random_state=RS,
        )
    ),
}